# Notebook 05 — Explainability (SHAP, LIME) and Fairness

### What this notebook does
| Step | What | Reviewer comment |
|---|---|---|
| 1 | **SHAP** for the feature set B LightGBM model, on the correct test set | #8 |
| 2 | Cross-check against **Logistic Regression** (linear SHAP + odds ratios) | #8 |
| 3 | **SHAP stability**: retrain on 5 different halves of the training set — does the ranking hold? | #8, #9 |
| 4 | **LIME** for 200 test people, with a **fidelity** check and agreement with SHAP | #8, #9 |
| 5 | **Fairness**: AUC, calibration, sensitivity and false-positive rate by age group, sex and race/ethnicity | #9 |

**Why LightGBM and not Stacking?** In Notebook 04 the two were statistically identical (DeLong p = 0.76).
LightGBM is a single tree model, so its SHAP values are computed **exactly** (TreeSHAP); Stacking would need
a slow approximation. The fairness analysis (Step 5) uses the pre-specified main model from Notebook 04.

**What changed from the old Notebook 10:** the old SHAP/LIME ran on rows that were partly *training* respondents
(the split was rebuilt wrongly), and the model contained post-stroke features. Everything here uses the true test
set and feature set B.

**Input** — `brfss_model_ready_v2.csv`, `feature_config_v2.json`, `models/v2/primary_B_*.joblib`,
`outputs/v2_thresholds.json`, `outputs/v2_test_predictions.csv` (from Notebook 04)
**Output** — `outputs/v2_shap_*.csv`, `v2_lr_odds_ratios.csv`, `v2_lime_*.csv`, `v2_fairness*.csv`,
`outputs/figures/fig_shap_*.png`, `fig_fairness_*.png`

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'lightgbm', 'shap', 'lime', '-q'])

import pandas as pd
import numpy as np
import os, json, time, joblib, itertools
import warnings
warnings.filterwarnings('ignore')
import matplotlib.pyplot as plt

import shap
from lime.lime_tabular import LimeTabularExplainer
from scipy.stats import spearmanr, rankdata
from sklearn.base import clone
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
import sklearn
print(f"scikit-learn {sklearn.__version__} | shap {shap.__version__}")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

BASE_DIR     = '/content/drive/MyDrive/stroke_prediction'
READY_PATH   = os.path.join(BASE_DIR, 'data', 'processed', 'brfss_model_ready_v2.csv')
CONFIG_PATH  = os.path.join(BASE_DIR, 'outputs', 'feature_config_v2.json')
MODELS_DIR   = os.path.join(BASE_DIR, 'models', 'v2')
OUTPUTS_DIR  = os.path.join(BASE_DIR, 'outputs')
FIG_DIR      = os.path.join(OUTPUTS_DIR, 'figures')
DICT_PATH    = os.path.join(OUTPUTS_DIR, 'variable_dictionary.csv')
os.makedirs(FIG_DIR, exist_ok=True)

# ── Run settings ──────────────────────────────────────────────────────────
SAMPLE       = 'primary'
FS           = 'B'
SHAP_N       = 20_000       # test respondents explained with SHAP (stratified sample)
STAB_RUNS    = 5            # retrain LightGBM on this many random halves of the training set
STAB_N       = 5_000        # test respondents used to compare rankings between those runs
LIME_N       = 200          # test respondents explained with LIME (half stroke, half non-stroke)
LIME_SAMPLES = 2_000        # perturbed samples LIME draws per person
LIME_REPEAT  = 50           # people re-explained with a different random seed (LIME stability)
TOP          = 5            # "top features" size used for agreement checks
SEED         = 42

config   = json.load(open(CONFIG_PATH))
FEATURES = config[f'features_{FS}']
df = pd.read_csv(READY_PATH)
df = df[df['meta_keep_primary']]
train = df[df['split'] == 'train'].reset_index(drop=True)
test  = df[df['split'] == 'test'].reset_index(drop=True)

lgbm = joblib.load(os.path.join(MODELS_DIR, f'{SAMPLE}_{FS}_LightGBM.joblib'))
lr   = joblib.load(os.path.join(MODELS_DIR, f'{SAMPLE}_{FS}_LogisticRegression.joblib'))
thr_info = json.load(open(os.path.join(OUTPUTS_DIR, 'v2_thresholds.json')))
MAIN = thr_info['main_model']

LABEL = {f: f for f in FEATURES}
if os.path.exists(DICT_PATH):
    d = pd.read_csv(DICT_PATH)
    if {'feature', 'label'} <= set(d.columns):
        for f, l in zip(d['feature'], d['label']):
            if f in LABEL and isinstance(l, str) and l.strip():
                LABEL[f] = f"{f}: {l.strip()[:40]}"

def stratified_sample(d, n, seed):
    if n >= len(d):
        return d.copy()
    s, _ = train_test_split(d, train_size=n, stratify=d['stroke'], random_state=seed)
    return s.reset_index(drop=True)

print(f"Feature set {FS}: {len(FEATURES)} features | test {len(test):,} | main model (fairness): {MAIN}")

### Step 1 — SHAP for LightGBM

**SHAP in one sentence:** for each person, it splits the model's prediction into "how much each answer pushed
the risk up or down" compared with the average person. Values are in **log-odds** (the model's internal scale):
+0.5 means that answer raised this person's odds of stroke by a factor of e^0.5 ≈ 1.65.

One question can become several model columns (a "was missing" flag, or one column per category for marital
status). We **add those columns back together**, so every SHAP value belongs to an original survey question.

**Additivity check:** SHAP values plus the baseline must equal the model's output exactly — the notebook verifies it.

In [ ]:
def column_owners(prep, features):
    """Map every column the preprocessor outputs back to the survey question it came from."""
    names = prep.get_feature_names_out()
    owner = []
    for n in names:
        part = n.split('__', 1)[1].replace('missingindicator_', '')
        if part in features:
            owner.append(part)
            continue
        match = [f for f in features if part.startswith(f + '_')]
        if not match:
            raise ValueError(f"Cannot map column {n} to a feature")
        owner.append(max(match, key=len))
    return np.array(names), np.array(owner)

def per_feature(values, owner, features):
    """Sum column-level contributions into one value per original feature (rows = people)."""
    return pd.DataFrame(values).T.groupby(owner).sum().T.reindex(columns=features, fill_value=0.0)

prep_g, model_g = lgbm.named_steps['prep'], lgbm.named_steps['model']
names_g, owner_g = column_owners(prep_g, FEATURES)

shap_people = stratified_sample(test, SHAP_N, SEED)
X_s = prep_g.transform(shap_people[FEATURES])
start = time.time()
contrib = model_g.booster_.predict(X_s, pred_contrib=True)        # exact TreeSHAP; last column = baseline
col_shap, base_value = contrib[:, :-1], contrib[0, -1]
feat_shap = per_feature(col_shap, owner_g, FEATURES)
gap = np.abs(contrib.sum(axis=1) - model_g.predict(X_s, raw_score=True)).max()
print(f"SHAP for {len(shap_people):,} test respondents in {time.time()-start:.0f}s | "
      f"{X_s.shape[1]} model columns -> {len(FEATURES)} features | additivity error {gap:.2e}")

def direction(f):
    if f in config['nominal']:
        return 'categorical'
    x = shap_people[f].to_numpy(float)
    ok = ~np.isnan(x)
    if ok.sum() < 30 or np.unique(x[ok]).size < 2:
        return 'n/a'
    rho = spearmanr(x[ok], feat_shap[f].to_numpy()[ok]).correlation
    return 'higher value -> higher risk' if rho > 0.1 else 'higher value -> lower risk' if rho < -0.1 else 'mixed'

imp = pd.DataFrame({'mean_abs_shap': feat_shap.abs().mean()}).sort_values('mean_abs_shap', ascending=False)
imp['rank'] = np.arange(1, len(imp) + 1)
imp['share_pct'] = 100 * imp['mean_abs_shap'] / imp['mean_abs_shap'].sum()
imp['direction'] = [direction(f) for f in imp.index]
imp['label'] = [LABEL[f] for f in imp.index]
imp.round(4).to_csv(os.path.join(OUTPUTS_DIR, 'v2_shap_importance.csv'), index_label='feature')
print(imp.head(20).round(4).to_string())

In [ ]:
top20 = imp.head(20)
fig, ax = plt.subplots(figsize=(8, 6.5))
ax.barh(range(len(top20))[::-1], top20['mean_abs_shap'], color='#3b6ea5')
ax.set_yticks(range(len(top20))[::-1], top20['label'], fontsize=8)
ax.set_xlabel('Mean |SHAP value| (log-odds)')
ax.set_title(f'Global feature importance — LightGBM, feature set {FS}, test set (n = {len(shap_people):,})', fontsize=9)
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, 'fig_shap_bar.png'), dpi=300)
plt.show()

top15 = list(imp.index[:15])
shap.summary_plot(feat_shap[top15].to_numpy(), shap_people[top15].to_numpy(float),
                  feature_names=[LABEL[f] for f in top15], max_display=15, show=False, plot_size=(9, 6.5))
plt.title('SHAP values per respondent (colour = answer value; grey = missing)', fontsize=9)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'fig_shap_beeswarm.png'), dpi=300)
plt.show()
print("Note: for nominal features (e.g. marital status) the colour is just the category code, not 'low/high'.")

### Step 2 — Cross-check with Logistic Regression

If two very different models — a boosted tree and a linear model — agree on which questions matter, that is
evidence the ranking reflects the **data**, not a quirk of one algorithm.

For a linear model SHAP has an exact formula: weight × (value − average value). We rank features that way and
compare with LightGBM using **Spearman's rank correlation** (1 = identical order).

The **odds ratio** table is the classic epidemiology view: for scaled numeric columns it is "per 1 standard
deviation"; for yes/no columns "yes vs no". The L1 penalty sets weak columns to exactly 0 (OR = 1).

In [ ]:
prep_l, model_l = lr.named_steps['prep'], lr.named_steps['model']
names_l, owner_l = column_owners(prep_l, FEATURES)
X_l = prep_l.transform(shap_people[FEATURES])
coef = model_l.coef_[0]
lin_shap = per_feature((X_l - X_l.mean(axis=0)) * coef, owner_l, FEATURES)
lr_imp = lin_shap.abs().mean()

both = pd.DataFrame({'lgbm_rank': imp['rank'], 'lr_rank': lr_imp.rank(ascending=False, method='min').astype(int)})
rho = spearmanr(imp['mean_abs_shap'].reindex(FEATURES), lr_imp.reindex(FEATURES)).correlation
top10_overlap = len(set(imp.index[:10]) & set(lr_imp.nlargest(10).index))
print(f"Spearman rank correlation LightGBM vs Logistic Regression: {rho:.3f} | top-10 overlap {top10_overlap}/10\n")
print(both.sort_values('lgbm_rank').head(15).to_string())

orr = pd.DataFrame({'column': names_l, 'feature': owner_l, 'coef': coef, 'odds_ratio': np.exp(coef)})
orr['scale'] = np.where(orr['column'].str.startswith('num__') & ~orr['column'].str.contains('missingindicator'),
                        'per 1 SD', np.where(orr['column'].str.contains('missingindicator'), 'missing vs answered',
                                             'yes vs no / this category'))
orr = orr[orr['coef'] != 0].reindex(orr['coef'].abs().sort_values(ascending=False).index)
orr.round(4).to_csv(os.path.join(OUTPUTS_DIR, 'v2_lr_odds_ratios.csv'), index=False)
print(f"\nLogistic Regression: {len(orr)} of {len(coef)} columns kept by the L1 penalty. Top 15 by |weight|:")
print(orr.head(15).round(3).to_string(index=False))

### Step 3 — SHAP stability

Retrain LightGBM (same hyperparameters) on **5 different random halves** of the training set, recompute SHAP on
the same test people, and compare the rankings. If the top features keep their places, the explanation is a
property of the data; if they shuffle around, it should not be over-interpreted.

In [ ]:
stab_people = stratified_sample(test, STAB_N, SEED)
imp_runs = {'reference': None}
X_ref = prep_g.transform(stab_people[FEATURES])
imp_runs['reference'] = per_feature(model_g.booster_.predict(X_ref, pred_contrib=True)[:, :-1], owner_g, FEATURES).abs().mean()

for r in range(1, STAB_RUNS + 1):
    start = time.time()
    half = stratified_sample(train, len(train) // 2, SEED + r)
    pipe_r = clone(lgbm).fit(half[FEATURES], half['stroke'])
    _, owner_r = column_owners(pipe_r.named_steps['prep'], FEATURES)
    X_r = pipe_r.named_steps['prep'].transform(stab_people[FEATURES])
    imp_runs[f'run_{r}'] = per_feature(pipe_r.named_steps['model'].booster_.predict(X_r, pred_contrib=True)[:, :-1],
                                       owner_r, FEATURES).abs().mean()
    print(f"  run {r}: retrained on {len(half):,} in {(time.time()-start)/60:.1f} min")

imp_runs = pd.DataFrame(imp_runs)
ranks = imp_runs.rank(ascending=False, method='min').astype(int)
pair_rho = [spearmanr(imp_runs[a], imp_runs[b]).correlation for a, b in itertools.combinations(imp_runs.columns, 2)]
top10_sets = [set(imp_runs[c].nlargest(10).index) for c in imp_runs.columns]
stab = pd.DataFrame({'rank_reference': ranks['reference'], 'rank_min': ranks.min(axis=1), 'rank_max': ranks.max(axis=1),
                     'in_top10_runs': [sum(f in s for s in top10_sets) for f in FEATURES]}, index=FEATURES)
stab = stab.sort_values('rank_reference')
stab.to_csv(os.path.join(OUTPUTS_DIR, 'v2_shap_stability.csv'), index_label='feature')
print(f"\nMean pairwise Spearman between {len(imp_runs.columns)} runs: {np.mean(pair_rho):.3f} (min {np.min(pair_rho):.3f})")
print(f"Features in the top 10 of every run: {int((stab['in_top10_runs'] == len(top10_sets)).sum())}")
print(stab.head(15).to_string())

### Step 4 — LIME, with a fidelity check

**LIME** explains one person at a time: it makes thousands of slightly changed copies of that person, asks the
model about each, and fits a small linear model to the answers. Two honest checks reviewers asked for:

* **Fidelity** — how well does LIME's little linear model reproduce the real model near this person?
  (R² of the local fit, and the gap between LIME's predicted probability and the model's real one.)
* **Stability** — explain the same person again with a different random seed: are the top features the same?

We also check **agreement with SHAP**: how many of each person's top-5 LIME features are in their top-5 SHAP features.

In [ ]:
y_s = shap_people['stroke'].to_numpy()
rng = np.random.default_rng(SEED)
pos = rng.choice(np.where(y_s == 1)[0], min(LIME_N // 2, (y_s == 1).sum()), replace=False)
neg = rng.choice(np.where(y_s == 0)[0], min(LIME_N - len(pos), (y_s == 0).sum()), replace=False)
people = np.concatenate([pos, neg])

background = prep_g.transform(stratified_sample(train, 5_000, SEED)[FEATURES])
binary_cols = [j for j in range(background.shape[1]) if set(np.unique(background[:, j])) <= {0.0, 1.0}]

def make_explainer(seed):
    return LimeTabularExplainer(background, feature_names=list(names_g), class_names=['no stroke', 'stroke'],
                                categorical_features=binary_cols, discretize_continuous=True,
                                mode='classification', random_state=seed)

def lime_top(exp):
    w = pd.Series({j: v for j, v in exp.as_map()[1]})
    return set(w.abs().groupby(owner_g[w.index]).sum().nlargest(TOP).index)

explainer = make_explainer(SEED)
start = time.time()
lime_rows, first_tops = [], {}
for i in people:
    exp = explainer.explain_instance(X_s[i], model_g.predict_proba, num_features=10, num_samples=LIME_SAMPLES)
    p_model = model_g.predict_proba(X_s[i:i + 1])[0, 1]
    top_lime = lime_top(exp)
    top_shap = set(feat_shap.iloc[i].abs().nlargest(TOP).index)
    first_tops[i] = top_lime
    lime_rows.append({'person': int(i), 'stroke': int(y_s[i]), 'model_prob': p_model,
                      'lime_local_prob': float(exp.local_pred[0]), 'lime_r2': exp.score,
                      'abs_prob_gap': abs(p_model - float(exp.local_pred[0])),
                      f'top{TOP}_overlap_with_shap': len(top_lime & top_shap) / TOP,
                      'lime_top': ', '.join(sorted(top_lime))})
print(f"LIME for {len(people)} people in {(time.time()-start)/60:.1f} min")

explainer2 = make_explainer(SEED + 1)
repeat = []
for i in people[:LIME_REPEAT]:
    exp = explainer2.explain_instance(X_s[i], model_g.predict_proba, num_features=10, num_samples=LIME_SAMPLES)
    repeat.append(len(lime_top(exp) & first_tops[i]) / TOP)

lime_df = pd.DataFrame(lime_rows)
lime_df.round(4).to_csv(os.path.join(OUTPUTS_DIR, 'v2_lime_explanations.csv'), index=False)
summary = pd.DataFrame({
    'measure': ['LIME local R² (median)', 'LIME local R² (IQR)', '|model prob − LIME prob| (median)',
                f'Top-{TOP} overlap LIME vs SHAP (mean)', f'Top-{TOP} overlap LIME seed 1 vs seed 2 (mean)'],
    'value': [f"{lime_df['lime_r2'].median():.3f}",
              f"{lime_df['lime_r2'].quantile(.25):.3f}–{lime_df['lime_r2'].quantile(.75):.3f}",
              f"{lime_df['abs_prob_gap'].median():.3f}",
              f"{lime_df[f'top{TOP}_overlap_with_shap'].mean():.2f}", f"{np.mean(repeat):.2f}"]})
summary.to_csv(os.path.join(OUTPUTS_DIR, 'v2_lime_fidelity.csv'), index=False)
print(summary.to_string(index=False))
print("\nMost frequent LIME top features:")
print(pd.Series(', '.join(lime_df['lime_top']).split(', ')).value_counts().head(10).to_string())

### Step 5 — Fairness by age, sex and race/ethnicity (reviewer comment #9)

For each group we ask four questions about the **main model** (calibrated, at the Youden threshold from Notebook 04):

| Question | Measure |
|---|---|
| Does it rank people as well in this group? | AUC (95% CI) |
| Are its probabilities right for this group? | **O/E** = observed ÷ expected cases (1 = right; > 1 = risk under-estimated) |
| Does it catch stroke cases equally? | Sensitivity (**equal opportunity**) |
| Does it raise false alarms equally? | False-positive rate (with sensitivity: **equalised odds**) |

**Read the age results carefully:** age is the strongest predictor, so *within* one age band the model loses its
best clue and AUC is naturally lower. That is expected, not unfairness. Groups with fewer than 20 stroke cases are
not reported (too few to say anything).

In [ ]:
def delong_auc(y, p):
    pos, neg = p[y == 1], p[y == 0]
    m, n = len(pos), len(neg)
    tz = rankdata(np.concatenate([pos, neg]))
    auc = tz[:m].sum() / (m * n) - (m + 1) / (2 * n)
    v01 = (tz[:m] - rankdata(pos)) / n
    v10 = 1 - (tz[m:] - rankdata(neg)) / m
    se = np.sqrt(np.var(v01, ddof=1) / m + np.var(v10, ddof=1) / n)
    return auc, auc - 1.96 * se, auc + 1.96 * se

pred = pd.read_csv(os.path.join(OUTPUTS_DIR, 'v2_test_predictions.csv'))
p_main = pred[f'p_{MAIN}'].to_numpy()
t_main = thr_info['thresholds'][MAIN]['youden']
y_t = pred['stroke'].to_numpy()

groups = {
    'Age group': pd.cut(pred['age'], [17, 44, 64, 79, 200], labels=['18–44', '45–64', '65–79', '80+']).astype(str),
    'Sex': pred['gender'].map({1: 'Male', 0: 'Female'}).fillna('Unknown'),
    'Race/ethnicity': pred['meta_race'].map({1: 'White, non-Hispanic', 2: 'Black, non-Hispanic',
                                             3: 'Other race, non-Hispanic', 4: 'Multiracial, non-Hispanic',
                                             5: 'Hispanic'}).fillna('Unknown/refused'),
}

def group_row(attr, level, mask):
    y, p = y_t[mask], p_main[mask]
    flag = p >= t_main
    auc, lo_ci, hi_ci = delong_auc(y, p)
    return {'attribute': attr, 'group': level, 'n': int(mask.sum()), 'stroke_cases': int(y.sum()),
            'prevalence_pct': 100 * y.mean(), 'auc': auc, 'auc_low': lo_ci, 'auc_high': hi_ci,
            'mean_predicted_pct': 100 * p.mean(), 'o_e_ratio': y.sum() / p.sum(),
            'sensitivity': flag[y == 1].mean(), 'false_positive_rate': flag[y == 0].mean(),
            'ppv': y[flag].mean() if flag.any() else np.nan, 'flagged_pct': 100 * flag.mean()}

rows = [group_row('Overall', 'All', np.ones(len(y_t), bool))]
for attr, g in groups.items():
    for level in sorted(g.unique()):
        mask = (g == level).to_numpy()
        if y_t[mask].sum() >= 20 and (y_t[mask] == 0).sum() >= 20:
            rows.append(group_row(attr, level, mask))
fair = pd.DataFrame(rows)
fair.round(4).to_csv(os.path.join(OUTPUTS_DIR, 'v2_fairness.csv'), index=False)

gaps = (fair[fair['attribute'] != 'Overall'].groupby('attribute', sort=False)
        .agg(auc_range=('auc', lambda s: s.max() - s.min()),
             o_e_min=('o_e_ratio', 'min'), o_e_max=('o_e_ratio', 'max'),
             sensitivity_gap=('sensitivity', lambda s: s.max() - s.min()),
             fpr_gap=('false_positive_rate', lambda s: s.max() - s.min())))
gaps.round(4).to_csv(os.path.join(OUTPUTS_DIR, 'v2_fairness_gaps.csv'))

show = fair.copy()
show['AUC (95% CI)'] = [f"{a:.3f} ({l:.3f}–{h:.3f})" for a, l, h in zip(fair['auc'], fair['auc_low'], fair['auc_high'])]
cols = ['attribute', 'group', 'n', 'stroke_cases', 'prevalence_pct', 'AUC (95% CI)', 'mean_predicted_pct',
        'o_e_ratio', 'sensitivity', 'false_positive_rate', 'ppv']
print(f"Main model: {MAIN} | Youden threshold {t_main:.4f}\n")
with pd.option_context('display.width', 250, 'display.float_format', '{:.3f}'.format):
    print(show[cols].to_string(index=False))
    print("\nGaps within each attribute (max − min across groups):")
    print(gaps.to_string())

In [ ]:
sub = fair[fair['attribute'] != 'Overall'].reset_index(drop=True)
overall = fair.iloc[0]
ypos = np.arange(len(sub))[::-1]
labels = [f"{a}: {g}" for a, g in zip(sub['attribute'], sub['group'])]

fig, axes = plt.subplots(1, 3, figsize=(13, 0.35 * len(sub) + 1.8), sharey=True)
axes[0].errorbar(sub['auc'], ypos, xerr=[sub['auc'] - sub['auc_low'], sub['auc_high'] - sub['auc']],
                 fmt='o', color='#3b6ea5', capsize=2)
axes[0].axvline(overall['auc'], color='grey', ls='--', lw=0.8)
axes[0].set(xlabel='AUC (95% CI)', title='Discrimination')
axes[1].plot(sub['o_e_ratio'], ypos, 'o', color='#b5532c')
axes[1].axvline(1, color='grey', ls='--', lw=0.8)
axes[1].set(xlabel='Observed / expected cases', title='Calibration')
axes[2].plot(sub['sensitivity'], ypos, 'o', label='Sensitivity', color='#2e7d4f')
axes[2].plot(sub['false_positive_rate'], ypos, 's', label='False-positive rate', color='#8a3b8f')
axes[2].set(xlabel='Rate at Youden threshold', title='Error rates')
axes[2].legend(fontsize=7, loc='lower right')
axes[0].set_yticks(ypos, labels, fontsize=8)
fig.suptitle(f'Subgroup performance — {MAIN}, test set', fontsize=10)
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, 'fig_fairness.png'), dpi=300)
plt.show()

print("=" * 70)
print("  Notebook 05 complete")
print("=" * 70)
print(f"  Top 5 SHAP features      : {', '.join(imp.index[:5])}")
print(f"  LightGBM vs LR (Spearman): {rho:.3f} | SHAP stability (mean Spearman) {np.mean(pair_rho):.3f}")
print(f"  LIME median R²           : {lime_df['lime_r2'].median():.3f} | LIME-SHAP top-{TOP} overlap "
      f"{lime_df[f'top{TOP}_overlap_with_shap'].mean():.2f} | LIME seed stability {np.mean(repeat):.2f}")
print(f"  Fairness tables          : outputs/v2_fairness.csv, v2_fairness_gaps.csv")
print(f"  Figures (300 DPI)        : fig_shap_bar.png, fig_shap_beeswarm.png, fig_fairness.png")
print("=" * 70)